# Prophet — Store Sales Forecasting

Prophet (Meta-ს) კლასიკური decomposition მოდელია:

`y(t) = trend(t) + seasonality(t) + holidays(t) + error`

trend-ს აღწერს ტეხილი წრფით (changepoint-ებით), სეზონურობას — Fourier სერიებით,
დღესასწაულებს — ცალკე რეგრესორებით. ის **ერთ სერიაზეა** გათვლილი, ამიტომ თითო
Store×Dept-ზე ცალკე Prophet-ს ვაწყობ (~3300 სერია).

### 1. ბიბლიოთეკები

In [ ]:
import sys
import os
import warnings
import logging

sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore")
os.environ.setdefault("WANDB_SILENT", "true")

# Prophet/Stan ბევრ ლოგს ბეჭდავს — ჩავახშობ
for logger_name in ["prophet", "cmdstanpy"]:
    logging.getLogger(logger_name).setLevel(logging.CRITICAL)

from src.data import load_raw
from src.metrics import wmae
from src.pipeline import RAW_COLS
from src.prophet_model import build_prophet_pipeline
from src.validation import time_holdout_split
from src.wandb_utils import init_run, log_pipeline

### 2. მონაცემები და ვალიდაცია

In [ ]:
raw = load_raw("data")
train = raw.train

tr, val = time_holdout_split(train, n_val_weeks=12)
val = val.reset_index(drop=True)

print("train:", tr.shape)
print("validation:", val.shape)

### 3. ვარიანტები

ვცვლი: სეზონურობის ტიპს (additive/multiplicative), trend-ის მოქნილობას
(`changepoint_prior_scale`), დღესასწაულების ჩართვას და კვირის სეზონურობას.

In [ ]:
EXPERIMENTS = [
    {
        "name": "Prophet_v1_additive_holidays",
        "cfg": {"seasonality_mode": "additive", "changepoint_prior_scale": 0.05,
                "use_holidays": True},
    },
    {
        "name": "Prophet_v2_multiplicative",
        "cfg": {"seasonality_mode": "multiplicative", "changepoint_prior_scale": 0.05,
                "use_holidays": True},
    },
    {
        "name": "Prophet_v3_flexible_trend",
        "cfg": {"seasonality_mode": "additive", "changepoint_prior_scale": 0.5,
                "use_holidays": True},
    },
    {
        "name": "Prophet_v4_no_holidays",
        "cfg": {"seasonality_mode": "additive", "changepoint_prior_scale": 0.05,
                "use_holidays": False},
    },
    {
        "name": "Prophet_v5_weekly_seasonality",
        "cfg": {"seasonality_mode": "additive", "changepoint_prior_scale": 0.1,
                "use_holidays": True, "weekly_seasonality": True},
    },
]

print("სულ ვარიანტი:", len(EXPERIMENTS))

### 4. თითო ვარიანტის გაშვება

თითო ვარიანტზე ~3300 Prophet ფიტდება (joblib-ით პარალელურად, `ProphetForecaster`-ის
შიგნით), ამიტომ ცოტა ხანს იღებს.

In [ ]:
results = []

for exp in EXPERIMENTS:
    name = exp["name"]
    cfg = exp["cfg"]

    run = init_run(group="Prophet_Training", job_type="experiment", name=name, config=cfg)

    pipe = build_prophet_pipeline(**cfg)
    pipe.fit(tr[RAW_COLS], tr["Weekly_Sales"])

    pred = pipe.predict(val[RAW_COLS])
    score = wmae(val["Weekly_Sales"], pred, val["IsHoliday"])

    run.summary["holdout_wmae"] = score
    run.summary["wmae_val"] = score
    log_pipeline(run, pipe, name="walmart_prophet_" + name.split("_", 1)[1],
                 metadata={"holdout_wmae": score})
    run.finish()

    results.append((name, score))
    print(name, "->", round(score, 2), "WMAE")

### 5. საუკეთესო + რეგისტრაცია

In [ ]:
best_name = None
best_score = float("inf")

for name, score in results:
    if score < best_score:
        best_score = score
        best_name = name

best_cfg = None
for exp in EXPERIMENTS:
    if exp["name"] == best_name:
        best_cfg = exp["cfg"]
        break

print("საუკეთესო:", best_name, "->", round(best_score, 2))

In [ ]:
run = init_run(group="Prophet_Training", job_type="final", name="Prophet_Final", config=best_cfg)

final_pipe = build_prophet_pipeline(**best_cfg)
final_pipe.fit(train[RAW_COLS], train["Weekly_Sales"])

run.summary["holdout_wmae"] = best_score
run.summary["wmae_val"] = best_score
log_pipeline(run, final_pipe, name="walmart_prophet",
             metadata={"holdout_wmae": best_score}, aliases=["best"])
run.finish()
print("დარეგისტრირდა: walmart_prophet:best")

### შედეგები

| ვარიანტი | WMAE |
|---|---|
| v1 additive + holidays | 1469 |
| v2 multiplicative | 1507 |
| v3 flexible trend | 1474 |
| v4 no holidays | 1567 |
| **v5 + weekly seasonality** | **1467** |

Prophet ბევრად ჯობია XGBoost-ს (1467 vs 1869), რადგან პირდაპირ სწავლობს თითო სერიის
სეზონურ ფორმას. holidays-ის მოშორება აუარესებს (v4=1567) — ე.ი. დღესასწაულების
რეგრესორები მართლა ეხმარება.